# Set quality gates for a support bot

Ridgeline Outfitters wants two regressions to stop a release: quoting the wrong catalog price and including an email address in a public product reply. Start with the kit’s recommendation, define the two checks precisely, and export them for CI. All examples are synthetic; no provider account is required.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7"], check=True)

## Get a recommendation from your app description

In [ ]:
from fi.evals.autoeval import AutoEvalConfig, AutoEvalPipeline, EvalConfig, ScannerConfig, list_templates

proposal = AutoEvalPipeline.from_description(
    "A customer support chatbot retrieves product descriptions and catalog prices. "
    "It replies to shoppers in a public chat, where emails must not be displayed."
)
print(proposal.explain())
print("Available templates:", ", ".join(sorted(list_templates())))

## Turn the recommendation into release criteria

A general grounding score does not guarantee an exact price. Compare the quoted price to an independent catalog value. Scan the whole reply for email-shaped strings; that rule is deliberately narrower than detecting all personal information.

In [ ]:
pipeline = AutoEvalPipeline.from_config(AutoEvalConfig(
    name="public_product_quotes",
    description="Exact catalog price and no email addresses in public product replies",
    evaluations=[EvalConfig(name="equals", threshold=1.0)],
    scanners=[ScannerConfig(name="RegexScanner", action="block", params={"patterns": ["email"]})],
    global_pass_rate=1.0,
))
print(pipeline.summary())

## Run the gates on three replies

The catalog has one USD price per item. Require exactly one quoted price in the format used by this support widget; do not use this extractor for a multi-currency cart.

In [ ]:
import re

catalog = {"trail_mug": "$18.00"}
replies = {
    "correct": "The trail mug costs $18.00.",
    "wrong price": "The trail mug costs $28.00.",
    "contact disclosed": "The trail mug costs $18.00. Ask shopper@example.invalid about it.",
}

def check_quote(pipe, reply, expected):
    prices = re.findall(r"\$[\d,]+(?:\.\d{2})?(?![\d.]*\d)", reply)
    result = pipe.evaluate({"output": " | ".join(prices), "expected_output": expected}, scan_content=reply)
    scan = result.scan_result
    if not scan or [s.scanner_name for s in scan.results] != ["regex"]:
        raise RuntimeError("Email scanner did not run")
    if any((s.reason or "").startswith("Scanner error:") for s in scan.results):
        raise RuntimeError("Email scanner failed")
    if result.blocked_by_scanner:
        return "blocked by email rule"
    metrics = result.metric_results
    if len(metrics) != 1 or metrics[0].eval_name != "equals" or metrics[0].status != "completed" or metrics[0].score is None:
        raise RuntimeError("Missing price check")
    return "pass" if result.passed else "price mismatch"

observed = {name: check_quote(pipeline, reply, catalog["trail_mug"]) for name, reply in replies.items()}
for name, verdict in observed.items():
    print(f"{name}: {verdict}")
assert list(observed.values()) == ["pass", "price mismatch", "blocked by email rule"]

## Find the failure pattern

In [ ]:
from collections import Counter
for reason, count in Counter(observed.values()).items():
    print(f"{count}/{len(observed)}: {reason}")

## Export and replay the same configuration

Save the evaluator configuration alongside the price extractor and regression cases. The configuration alone does not contain the catalog, the extractor, or your test inputs.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from fi.evals.autoeval import load_json

out = Path(mkdtemp(prefix="quote-gates-"))
yaml_path, json_path = out / "pipeline.yaml", out / "pipeline.json"
pipeline.export_yaml(str(yaml_path))
pipeline.export_json(str(json_path))
reloaded = AutoEvalPipeline.from_yaml(str(yaml_path))
assert reloaded.config.to_dict() == load_json(str(json_path)).to_dict() == pipeline.config.to_dict()
assert {name: check_quote(reloaded, reply, catalog["trail_mug"]) for name, reply in replies.items()} == observed
print(f"Reloaded both configurations and replayed all three cases. Files: {out}")